Setting Up the environment  
(1) Open a folder --> CMD: mkdir folder_name  
(2) Access the folder --> CMD: cd folder_name  
(3) Create Virtual Environment --> CMD: python -m venv env_name  
(4) Activate the Environment --> CMD: env\Scripts\activate  
(5) Put all your packages in a requirements.txt file and use CMD: pip install -r requirements.txt  
(6) Open a python file  
(7) Create a file named .env and inside it put the API key for the model  
For example:  GEMINI_API = "iuehquhe22781y7...." something like that

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
import os
load_dotenv()
GEMINI_API = os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = GEMINI_API,
    temperature = 0.3 
)

Testing the model responses with different system messages

In [3]:
messages = [
    SystemMessage(content = "You are a concise Python tutor."),
    HumanMessage(content = "Explain what recursion is in one short paragraph.")
]

response = model.invoke(messages)
print(response.content)

Recursion is a programming technique where a function calls itself to solve a problem. It works by breaking down a complex problem into smaller, identical subproblems until it reaches a simple "base case" that can be solved directly. Each recursive call then builds upon the solution of the previous one, eventually combining to solve the original problem.


In [4]:
messages = [
    SystemMessage(content = "You are a strict university professor. Explain with technical depth."),
    HumanMessage(content = "Explain what recursion is in one short paragraph.")
]

response = model.invoke(messages)
print(response.content)

Recursion is a computational paradigm where a function defines itself in terms of its own application, solving a problem by decomposing it into one or more identical, yet simpler, subproblems. This self-referential process continues until a non-recursive "base case" is reached, which provides a direct solution without further recursion. The results are then progressively combined as the execution unwinds from the call stack, with each preceding recursive call integrating the solution of its subproblem(s) until the initial problem is fully resolved. Crucially, the absence of a properly defined base case inevitably leads to infinite recursion and a stack overflow error.


Prompt Templates

In [5]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words for a {level} learner."
)
formatted_prompt = prompt.invoke({
    "topic":"recursion",
    "level":"beginner"
})

print(formatted_prompt)

text='Explain recursion in simple words for a beginner learner.'


Chat Prompt Template

In [14]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise tutor. Always explain clearly."),
    ("user", "Explain {topic} in simple words for a {level} learner.")
])
formatted_chat_prompt = prompt.invoke({
    "topic":"recursion",
    "level":"beginner"
})
print(formatted_chat_prompt)

messages=[SystemMessage(content='You are a concise tutor. Always explain clearly.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain recursion in simple words for a beginner learner.', additional_kwargs={}, response_metadata={})]


Adding Context Around The Prompt

In [15]:
from langchain_core.output_parsers import StrOutputParser

context_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a clear technical tutor.

Follow these rules:
- Match the explanation to the learner level.
- Respect the requested style.
- Do not add unnecessary details.
"""),
    ("human", """
Topic:
{topic}

Learner level:
{learner_level}

Goal:
{goal}

Style:
{style}

Explain the topic based on this context.
""")
])

parser = StrOutputParser()

context_chain = context_prompt | model | parser

In [16]:
context_chain

ChatPromptTemplate(input_variables=['goal', 'learner_level', 'style', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\nYou are a clear technical tutor.\n\nFollow these rules:\n- Match the explanation to the learner level.\n- Respect the requested style.\n- Do not add unnecessary details.\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['goal', 'learner_level', 'style', 'topic'], input_types={}, partial_variables={}, template='\nTopic:\n{topic}\n\nLearner level:\n{learner_level}\n\nGoal:\n{goal}\n\nStyle:\n{style}\n\nExplain the topic based on this context.\n'), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.7'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_up

context_prompt | model | parser This Structure called the LCEL 

In [17]:
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical tutor."),
    ("human", """
Explain the following topic clearly.

Topic:
{topic}

Audience:
{audience}

Keep the explanation short and practical.
""")
])

parser = StrOutputParser()

explain_chain = explain_prompt | model | parser

In [18]:
result = explain_chain.invoke({
    "topic": "recursion",
    "audience": "beginner Python learner"
})

print(result)

Recursion is when a function calls *itself* to solve a problem.

**Core Idea:** You break a big problem into smaller, identical versions of itself until it's simple enough to solve directly.

**Two Key Parts:**

1.  **Base Case:** This is the "stop" condition. It's the simplest version of the problem that the function can solve without calling itself again. **Crucial to prevent infinite loops!**
2.  **Recursive Step:** The function calls itself with a modified input that moves closer to the base case.

**Practical Example (Factorial `n! = n * (n-1)!`):**

To calculate `factorial(5)`:
*   It asks for `5 * factorial(4)`
*   `factorial(4)` asks for `4 * factorial(3)`
*   ...
*   `factorial(1)` asks for `1 * factorial(0)`
*   `factorial(0)` is the **base case** (it returns `1` directly).
*   Then, the results multiply back up: `1 * 1 = 1`, then `2 * 1 = 2`, then `3 * 2 = 6`, etc.

**When to Use:** It's practical for problems that naturally define themselves in terms of smaller versions of 

The LCEL is used to avoid manually calling the langchain runnable objects  
so rather than invoking the prompt, model, and parser manually you do it all together using the LCEL  
  
To include an element in the chain it must be runnable, so if you intended to build a custom function  
you need to make it runnable, and this is done through something called runnable lambda

In [19]:
from langchain_core.runnables import RunnableLambda

def clean_topic_input(data: dict) -> dict:
    """
    Clean the topic and audience before sending them to the prompt.
    """
    return {
        "topic": data["topic"].strip(),
        "audience": data["audience"].strip()
    }

cleaner = RunnableLambda(clean_topic_input)

In [20]:
clean_explain_chain = cleaner | explain_prompt | model | parser

In [21]:
topics = [
    {"topic": "recursion", "audience": "beginner Python learner"},
    {"topic": "decorators", "audience": "intermediate Python learner"},
    {"topic": "generators", "audience": "advanced Python learner"},
]

results = clean_explain_chain.batch(topics)

for result in results:
    print("=" * 60)
    print(result)

Recursion is when a function calls itself to solve a problem.

Imagine you have a big task that can be solved by doing a slightly smaller version of the *exact same task*, until the task becomes so small it's trivial.

Every recursive function needs two things:
1.  **Base Case:** The simplest version of the problem, where the function stops calling itself. (Crucial to prevent infinite loops!)
2.  **Recursive Step:** The function calls itself with a modified input that moves closer to the base case.

**Practical Use:** It's elegant for problems naturally defined in terms of smaller, similar sub-problems (e.g., calculating factorials, traversing tree structures).

**Example: Factorial (n!)**
`5! = 5 * 4 * 3 * 2 * 1`. Notice `4! = 4 * 3 * 2 * 1`. So, `5! = 5 * 4!`

```python
def factorial(n):
    if n == 0:  # Base Case: Factorial of 0 is 1
        return 1
    else:       # Recursive Step: n! = n * (n-1)!
        return n * factorial(n-1)

print(factorial(5)) # Output: 120
```
A decorato

Parallel Chains

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You summarize technical topics in one sentence."),
    ("human", "Summarize this topic: {topic}")
])

example_prompt = ChatPromptTemplate.from_messages([
    ("system", "You create simple coding examples."),
    ("human", "Give a tiny example for this topic: {topic}")
])

summary_chain = summary_prompt | model | parser
example_chain = example_prompt | model | parser

parallel_chain = {
    "summary": summary_chain,
    "example": example_chain
}

result = parallel_chain.invoke({
    "topic": "recursion"
})

print(result)

Tools  
This is how we create tools in langchain, these tools will be evaluated later by the model whether to make the system executing them or not depending on the context

In [31]:
from langchain_core.tools import tool
@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

In [29]:
result = multiply_numbers.invoke({
    "a":5,
    "b":6
})
result

30

Creating Agents

In [32]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[multiply_numbers],
    system_prompt="You are a helpful technical assistant."
)
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 7 multiplied by 8?"
        }
    ]
})
result

{'messages': [HumanMessage(content='What is 7 multiplied by 8?', additional_kwargs={}, response_metadata={}, id='efa7217a-9b92-4381-8c5f-fc89bacffa28'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply_numbers', 'arguments': '{"a": 7, "b": 8}'}, '__gemini_function_call_thought_signatures__': {'585a3d5d-59b3-4cd7-84ae-1a3250803652': 'Cv0BARFNMg+lHgDt+wEqMvuDBzEI5wSzUDyoZ841UCgbP/IIgIUnL6STDGOOpqBZvX/Q3VAXWjcQV7iWVgJ25uyYubRkIRoxo4+N+P6McKu4js2WVS/P82uthqHL2QocDS0OGP1bkHrBw8mKHfMBbFlKZ8SI3KGCD0TrTBkFuT3ZOQOflomwnV8XQk78iMSSfS6zBHdZdba9jlofhn1m6mg8cJKjAVXo1pqs2DFLuiNv6Dg3P6LuvMlNd+2wm7nHNyToO2gRELRiB+6O/FcE+0wV8KTRyQjjtMiFl4hdewFgN14L2YKQ7Ze7IQ3AgzIRk76j/YvoUjOhvNOckb30+A=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f3d33-88a5-7181-93d8-32f507a6394d-0', tool_calls=[{'name': 'multiply_numbers', 'args': {'a': 7, 'b': 8}, 'id': '585a3d5d-59b3-4cd7-84ae-

Pydantic

In [35]:
from pydantic import BaseModel
from typing import Literal
class TaskSummary(BaseModel):
    task_type: str
    risk_level: Literal["low", "medium", "high"]
    summary: str
    next_steps: list[str]

valid_result = TaskSummary(
    task_type="debugging",
    risk_level="high",
    summary="The code has a possible runtime error because a variable name is incorrect.",
    next_steps=[
        "Check the variable names",
        "Replace the undefined variable",
        "Run a small test case"
    ]
)

print(valid_result)

task_type='debugging' risk_level='high' summary='The code has a possible runtime error because a variable name is incorrect.' next_steps=['Check the variable names', 'Replace the undefined variable', 'Run a small test case']


In [41]:
try:
    invalid_result = TaskSummary(
        task_type="debugging",
        risk_level="Dangerous",
        summary="The code has a possible runtime error because a variable name is incorrect.",
        next_steps=[
            "Check the variable names",
            "Replace the undefined variable",
            "Run a small test case"
        ]
    )
except Exception as e:
    print("Invalid Fields")
    print(e) 

Invalid Fields
1 validation error for TaskSummary
risk_level
  Input should be 'low', 'medium' or 'high' [type=literal_error, input_value='Dangerous', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


In [2]:
import parso

code = """
deff add(a, b):    # Error 1
    return a + b

ifff True:         # Error 2
    print("Hi")
    
10 = x             # Error 3
"""

# Parso builds the tree even if the code is completely broken
grammar = parso.load_grammar()
tree = grammar.parse(code)

# Now, we ask Parso to list all the syntax errors it found
errors = grammar.iter_errors(tree)

print("Errors found:")
for error in errors:
    print(f"- Line {error.start_pos[0]}: {error.message}")

Errors found:
- Line 2: SyntaxError: invalid syntax
- Line 3: IndentationError: unexpected indent
- Line 5: SyntaxError: invalid syntax
- Line 6: IndentationError: unexpected indent
- Line 8: SyntaxError: cannot assign to literal


In [1]:
from tools import check_python_syntax


bad_code = """
def hello(
    print("hi")
"""

result = check_python_syntax.invoke({
    "code": bad_code
})

print(result)

SyntaxError at line 3, column 9: SyntaxError: invalid syntax
